# Semana 5: SQL DML y JOINs
## Notebook de Ejercicios (NB2) – Práctica en DB-Fiddle

### Objetivo
Poner en práctica los conceptos de JOINs en un entorno SQL real utilizando **DB-Fiddle**. Escribiremos consultas con diferentes tipos de JOIN (INNER, LEFT, RIGHT, FULL) y resolveremos ejercicios que involucran múltiples tablas.

### Herramienta
Utilizaremos [DB-Fiddle](https://www.db-fiddle.com/) con motor **PostgreSQL**.

### Instrucciones generales
1. Abre [DB-Fiddle](https://www.db-fiddle.com/).
2. Selecciona **PostgreSQL** como motor.
3. En el panel superior (Schema), pega el código de creación de tablas e inserción de datos.
4. En el panel inferior (Query), escribe y ejecuta las consultas propuestas.
5. Observa los resultados y reflexiona sobre cada tipo de JOIN.

---
## 1. Creación de las Tablas de Ejemplo

Vamos a trabajar con un modelo de **clientes, pedidos y productos**. Un cliente puede tener varios pedidos, y un pedido puede contener varios productos (relación N:M a través de detalle_pedido).

### Código SQL para crear las tablas e insertar datos

Copia y pega el siguiente código en el panel **Schema** de DB-Fiddle y ejecútalo.

In [ ]:
# Código SQL para crear las tablas (cópialo y pégalo en DB-Fiddle)
sql_create = """
-- Tabla clientes
CREATE TABLE clientes (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE,
    ciudad VARCHAR(50)
);

-- Tabla productos
CREATE TABLE productos (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    precio DECIMAL(10,2) NOT NULL
);

-- Tabla pedidos (cabecera)
CREATE TABLE pedidos (
    id SERIAL PRIMARY KEY,
    fecha DATE NOT NULL DEFAULT CURRENT_DATE,
    cliente_id INT NOT NULL REFERENCES clientes(id)
);

-- Tabla detalle_pedido (líneas de pedido)
CREATE TABLE detalle_pedido (
    pedido_id INT REFERENCES pedidos(id),
    producto_id INT REFERENCES productos(id),
    cantidad INT NOT NULL CHECK (cantidad > 0),
    precio_unitario DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (pedido_id, producto_id)
);

-- Insertar datos de ejemplo
INSERT INTO clientes (nombre, email, ciudad) VALUES
    ('Juan Pérez', 'juan@email.com', 'Madrid'),
    ('María García', 'maria@email.com', 'Barcelona'),
    ('Carlos López', 'carlos@email.com', 'Valencia'),
    ('Ana Martínez', 'ana@email.com', 'Sevilla');

INSERT INTO productos (nombre, precio) VALUES
    ('Laptop', 1200.00),
    ('Mouse', 25.50),
    ('Teclado', 45.00),
    ('Monitor', 300.00),
    ('Auriculares', 80.00);

INSERT INTO pedidos (fecha, cliente_id) VALUES
    ('2025-03-01', 1),
    ('2025-03-02', 2),
    ('2025-03-02', 1),
    ('2025-03-03', 3),
    ('2025-03-04', 1),
    ('2025-03-05', 5); -- cliente 5 no existe

INSERT INTO detalle_pedido (pedido_id, producto_id, cantidad, precio_unitario) VALUES
    (1, 1, 1, 1200.00),
    (1, 2, 2, 25.50),
    (2, 3, 1, 45.00),
    (3, 4, 1, 300.00),
    (3, 5, 1, 80.00),
    (4, 1, 1, 1200.00),
    (4, 2, 1, 25.50),
    (5, 3, 2, 45.00);
"""
print("✅ Código SQL listo para copiar a DB-Fiddle.")

---
## 2. Ejercicios con JOINs

A continuación, escribe las consultas en el panel **Query** de DB-Fiddle y ejecútalas.

### Ejercicio 1: INNER JOIN

Obtén el listado de todos los pedidos con el nombre del cliente y la fecha del pedido.

**SQL esperado:**

In [ ]:
# Escribe esta consulta en DB-Fiddle
sql_ej1 = """
SELECT p.id, c.nombre AS cliente, p.fecha
FROM pedidos p
INNER JOIN clientes c ON p.cliente_id = c.id;
"""
print("✅ Consulta Ejercicio 1 lista.")

### Ejercicio 2: LEFT JOIN

Muestra todos los clientes, junto con sus pedidos (si tienen). Incluye los clientes sin pedidos. Ordena por cliente.

**SQL esperado:**

In [ ]:
sql_ej2 = """
SELECT c.id, c.nombre, p.id AS pedido_id, p.fecha
FROM clientes c
LEFT JOIN pedidos p ON c.id = p.cliente_id
ORDER BY c.nombre, p.fecha;
"""
print("✅ Consulta Ejercicio 2 lista.")

### Ejercicio 3: RIGHT JOIN

Muestra todos los pedidos, incluso aquellos cuyo cliente no existe (por ejemplo, cliente_id=5).

**SQL esperado:**

In [ ]:
sql_ej3 = """
SELECT p.id AS pedido_id, p.fecha, c.nombre AS cliente
FROM clientes c
RIGHT JOIN pedidos p ON c.id = p.cliente_id
ORDER BY p.id;
"""
print("✅ Consulta Ejercicio 3 lista.")

### Ejercicio 4: FULL OUTER JOIN

Combina clientes y pedidos para ver todos los clientes y todos los pedidos, incluso si no tienen correspondencia.

**SQL esperado:**

In [ ]:
sql_ej4 = """
SELECT c.id AS cliente_id, c.nombre, p.id AS pedido_id, p.fecha
FROM clientes c
FULL OUTER JOIN pedidos p ON c.id = p.cliente_id
ORDER BY c.id, p.id;
"""
print("✅ Consulta Ejercicio 4 lista.")

---
## 3. Ejercicios con Múltiples Tablas

### Ejercicio 5: JOIN de 3 tablas

Obtén el detalle completo de los pedidos: número de pedido, fecha, nombre del cliente, nombre del producto, cantidad y precio unitario.

**SQL esperado:**

In [ ]:
sql_ej5 = """
SELECT p.id AS pedido_id, p.fecha, c.nombre AS cliente,
       pr.nombre AS producto, dp.cantidad, dp.precio_unitario,
       (dp.cantidad * dp.precio_unitario) AS importe
FROM pedidos p
INNER JOIN clientes c ON p.cliente_id = c.id
INNER JOIN detalle_pedido dp ON p.id = dp.pedido_id
INNER JOIN productos pr ON dp.producto_id = pr.id
ORDER BY p.id, pr.nombre;
"""
print("✅ Consulta Ejercicio 5 lista.")

### Ejercicio 6: Clientes sin pedidos

Obtén los clientes que no han realizado ningún pedido.

**SQL esperado:**

In [ ]:
sql_ej6 = """
SELECT c.id, c.nombre
FROM clientes c
LEFT JOIN pedidos p ON c.id = p.cliente_id
WHERE p.id IS NULL;
"""
print("✅ Consulta Ejercicio 6 lista.")

### Ejercicio 7: Productos nunca vendidos

Encuentra los productos que nunca han aparecido en ningún detalle de pedido.

**SQL esperado:**

In [ ]:
sql_ej7 = """
SELECT pr.id, pr.nombre
FROM productos pr
LEFT JOIN detalle_pedido dp ON pr.id = dp.producto_id
WHERE dp.producto_id IS NULL;
"""
print("✅ Consulta Ejercicio 7 lista.")

### Ejercicio 8: Total gastado por cliente

Calcula el importe total gastado por cada cliente (suma de cantidad * precio_unitario). Muestra también los clientes que no han gastado nada (con total 0).

**SQL esperado:**

In [ ]:
sql_ej8 = """
SELECT c.id, c.nombre,
       COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0) AS total_gastado
FROM clientes c
LEFT JOIN pedidos p ON c.id = p.cliente_id
LEFT JOIN detalle_pedido dp ON p.id = dp.pedido_id
GROUP BY c.id, c.nombre
ORDER BY total_gastado DESC;
"""
print("✅ Consulta Ejercicio 8 lista.")

### Ejercicio 9: Pedidos con más de un producto

Encuentra los pedidos que tienen más de un producto (es decir, más de una fila en detalle_pedido).

**SQL esperado:**

In [ ]:
sql_ej9 = """
SELECT p.id, p.fecha, COUNT(dp.producto_id) AS num_productos
FROM pedidos p
INNER JOIN detalle_pedido dp ON p.id = dp.pedido_id
GROUP BY p.id, p.fecha
HAVING COUNT(dp.producto_id) > 1;
"""
print("✅ Consulta Ejercicio 9 lista.")

### Ejercicio 10: Self-Join (opcional)

Para practicar self-join, vamos a añadir una tabla `empleados` con estructura jerárquica (jefe - subordinado). Primero, crea la tabla con el siguiente código SQL en el panel Schema (después de las tablas anteriores, o reinicia el fiddle).

```sql
CREATE TABLE empleados (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100),
    jefe_id INT REFERENCES empleados(id)
);

INSERT INTO empleados (nombre, jefe_id) VALUES
    ('Ana Director', NULL),
    ('Carlos Manager', 1),
    ('Luis Subordinado', 2),
    ('Marta Subordinada', 2);
```

Luego, escribe una consulta que muestre cada empleado con el nombre de su jefe (self-join).

**SQL esperado:**

In [ ]:
sql_ej10 = """
SELECT e1.nombre AS empleado, e2.nombre AS jefe
FROM empleados e1
LEFT JOIN empleados e2 ON e1.jefe_id = e2.id
ORDER BY e1.id;
"""
print("✅ Consulta Ejercicio 10 lista.")

---
## 4. Reflexión y Análisis

Después de ejecutar las consultas, reflexiona sobre las siguientes preguntas:

1.  ¿Cuál es la diferencia práctica entre INNER JOIN y LEFT JOIN en los resultados?
2.  ¿Por qué en el Ejercicio 3 (RIGHT JOIN) obtuvimos un pedido con cliente NULL?
3.  En el Ejercicio 8, ¿por qué usamos COALESCE? ¿Qué pasaría si no lo usáramos?
4.  ¿Cómo cambiarían los resultados del Ejercicio 5 si usáramos LEFT JOIN en lugar de INNER JOIN en alguna de las uniones?

Escribe tus respuestas en una celda de texto.

In [ ]:
# Escribe aquí tus reflexiones
print("""
1. INNER JOIN muestra solo clientes con pedidos; LEFT JOIN muestra todos los clientes.
2. Porque hay un pedido con cliente_id=5 que no existe en clientes.
3. COALESCE convierte NULL en 0 para clientes sin pedidos. Sin él, aparecería NULL.
4. Si usáramos LEFT JOIN, aparecerían clientes sin pedidos con valores NULL en las columnas de pedidos.
""")

---
## Ejercicios para el Estudiante (Tarea)

1.  **Añade más datos:** Inserta nuevos clientes, productos y pedidos en DB-Fiddle y repite las consultas. Observa cómo cambian los resultados.

2.  **Crea tus propias consultas:** Inventa al menos 3 consultas adicionales que utilicen diferentes tipos de JOIN y comparte el código.

3.  **Analiza el rendimiento:** Para una de las consultas (por ejemplo, la del Ejercicio 5), usa `EXPLAIN` en DB-Fiddle (anteponiendo EXPLAIN a la consulta) y analiza el plan de ejecución.

4.  **Traduce a pandas:** Elige una de las consultas (por ejemplo, la del total gastado) y escribe el código Python con pandas que realizaría la misma operación (similar al NB1).

5.  **Reflexión final:** ¿En qué situaciones del mundo real usarías cada tipo de JOIN?

---
## Conclusiones de la Sesión

*   Hemos practicado los diferentes tipos de **JOIN** (INNER, LEFT, RIGHT, FULL) en un entorno SQL real.
*   Aprendimos a combinar múltiples tablas para obtener información completa.
*   Resolvimos problemas comunes como encontrar clientes sin pedidos o productos no vendidos.
*   Utilizamos funciones como `COALESCE` para manejar valores nulos.
*   Exploramos el self-join para relaciones jerárquicas.
*   Esta práctica consolida la comprensión de cómo las bases de datos relacionales integran información dispersa.

¡Ahora estás listo para enfrentarte a consultas SQL con múltiples tablas en cualquier entorno profesional!